# NT Syntactic Roles — Who Does What to Whom

Analysis of syntactic roles in the Greek NT using MACULA `subjref` links on
the Nestle1904 syntax tree. Each word token carries role, subjref, gloss,
domain, frame, tense, voice, mood, case, person, number, and gender.

This notebook covers loading NT syntax data, Jesus speaking verses,
subject/object search for God and Jesus, and syntactic role charts.

**Sections:**
1. Loading NT Greek Syntax Data (load_syntax, query_syntax)
2. Jesus Speaking Verses (jesus_speaking_verses, is_jesus_speaking)
3. What God (Theos) Does in the NT
4. What Jesus Does in the Gospels
5. Object/Argument Search (print_object_summary NT)
6. Syntactic Role Charts

In [ ]:
import sys
sys.path.insert(0, '../../../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print('Ready.')

---
## 1. Loading NT Greek Syntax Data

The `syntax` module wraps the MACULA Greek Nestle1904 syntax tree data.
Each word token carries:

- **`role`** — syntactic role: `s` (subject), `v` (verb), `o` (object), `io` (indirect object), `p` (predicate), `vc` (verb complement), `adv` (adverbial)
- **`subjref`** — xml_id of the grammatical subject of this verb
- **`referent`** — xml_id of the referent this word points to
- **`gloss`** — English gloss
- **`domain`** — Louw-Nida semantic domain
- **`frame`** — semantic frame label
- **`tense`, `voice`, `mood`, `case_`, `person`, `number`, `gender`** — full morphology

The first call to `load_syntax()` parses the TSV and writes a Parquet cache.

In [ ]:
from bible_grammar import load_syntax, query_syntax

df_nt = load_syntax()
print(f'NT syntax tokens: {len(df_nt):,}')
print(f'Columns: {list(df_nt.columns)}')

In [ ]:
# John 1:1 — "In the beginning was the Word"
jhn1 = query_syntax(book='Jhn', chapter=1, verse=1)
jhn1[['ref', 'text', 'lemma', 'strong_g', 'role', 'gloss', 'case_', 'tense']]

In [ ]:
# Distribution of syntactic roles across the NT
df_nt['role'].value_counts()

In [ ]:
# How many verb tokens have a resolved grammatical subject?
verbs = df_nt[df_nt['class_'] == 'verb']
has_subj = verbs['subjref'].notna().sum()
print(f'Total NT verb tokens:            {len(verbs):>7,}')
print(f'With resolved subject (subjref): {has_subj:>7,}  ({100*has_subj/len(verbs):.1f}%)')

---
## 2. Jesus Speaking Verses

The `speaker` module identifies NT verses where Jesus speaks using two
complementary strategies:
1. **Curated allowlists** — frozensets of exact (book, chapter, verse) triples
   for well-known verse sets (I AM sayings, Son of Man sayings, bridegroom parables)
2. **MACULA subjref detection** — speech verbs whose grammatical subject resolves to Iesous (G2424)

In [ ]:
from bible_grammar import speech_verbs, jesus_speaking_verses

# Find speech verbs in Matthew where Jesus is the grammatical subject
sv = speech_verbs('Mat', subject_strong='2424')   # G2424 = Iesous
print(f'Speech verbs with Jesus as subject in Matthew: {len(sv)}')
sv[['ref', 'text', 'lemma', 'gloss']].head(10)

In [ ]:
# Full set of verses where Jesus is the grammatical subject of a speech verb
gospel_books = ['Mat', 'Mrk', 'Luk', 'Jhn']
speaking = jesus_speaking_verses(gospel_books)
print(f'Distinct (book, chapter, verse) tuples where Jesus speaks: {len(speaking)}')

In [ ]:
from bible_grammar import jesus_speaking_verse_set, is_jesus_speaking, ALLOWLIST_VERSES

# All Gospel verses where Jesus speaks (MACULA detection)
speaking_set = jesus_speaking_verse_set(['Mat', 'Mrk', 'Luk', 'Jhn'])
print(f'Verses with Jesus as speaking subject (Gospels): {len(speaking_set)}')

# Curated allowlists
print('\nAllowlist titles:')
for title, verses in ALLOWLIST_VERSES.items():
    print(f'  {title}: {len(verses)} verses')

In [ ]:
# Check individual verses
test_verses = [
    ('Mat', 16, 13),   # "Who do people say the Son of Man is?" — Jesus speaking
    ('Jhn', 8, 58),    # "Before Abraham was born, I AM" — Johannine I AM
    ('Mat', 3, 17),    # "This is my Son" — God speaking, not Jesus
    ('Jhn', 11, 35),   # "Jesus wept" — narration, not speech
]

for book, ch, vs in test_verses:
    result = is_jesus_speaking(book, ch, vs)
    print(f'{book} {ch}:{vs:2d}  ->  Jesus speaking: {result}')

---
## 3. What God (Theos) Does in the NT

What verbs does Theos (G2316) take as grammatical subject in the NT?
This can be compared with YHWH+Elohim in the OT to trace continuity
and change in divine action language across the two Testaments.

In [ ]:
from bible_grammar import subject_verbs, print_role_summary

# What does God (Theos) do in the NT?
print_role_summary(['G2316'], corpus='NT', top_n=20, label='Theos')

---
## 4. What Jesus Does in the Gospels

What verbs does Iesous (G2424) take as grammatical subject in the Gospels?
This answers: 'What does Jesus *do* syntactically?' — combining healing verbs,
speech verbs, travel verbs, and authority verbs into one picture.

In [ ]:
# What does Jesus (Iesous) do in the Gospels?
print_role_summary(
    ['G2424'], corpus='NT', books=['Mat', 'Mrk', 'Luk', 'Jhn'],
    top_n=20, label='Iesous'
)

---
## 5. Object/Argument Search

Find what objects God and Jesus act upon in the NT.
The NT method finds verbs whose `subjref` resolves to the target entity,
then collects co-verse tokens tagged `role='o'` or `role='o2'`.

In [ ]:
from bible_grammar import print_object_summary

# What does Jesus act upon in the Gospels?
print_object_summary(
    ['G2424'], corpus='NT', books=['Mat', 'Mrk', 'Luk', 'Jhn'],
    top_n=20, label='Iesous (Gospels)'
)

---
## 6. Syntactic Role Charts

Visual outputs for syntactic role analysis.
The OT/NT comparison chart places God's verbs in both Testaments
side-by-side for theological and linguistic comparison.

In [ ]:
from bible_grammar import role_chart
from IPython.display import Image

# Chart: top verbs for Jesus in the Gospels
chart_path = role_chart(
    ['G2424'], corpus='NT', books=['Mat', 'Mrk', 'Luk', 'Jhn'],
    top_n=20, label='Iesous (Gospels)',
    output_path='../../../output/charts/nt/names/role-iesous-gospels.png'
)
print(f'Chart saved: {chart_path}')
Image(chart_path)

In [ ]:
from bible_grammar import divine_action_comparison
from IPython.display import Image

# Side-by-side: God's verbs in OT Hebrew vs NT Greek
ot_df, nt_df, chart_path = divine_action_comparison(
    output_path='../../../output/charts/both/names/divine-action-ot-nt.png'
)
print(f'Chart saved: {chart_path}')
Image(chart_path)